# Ablation: bo bot thanh phan khoi ham muc tieu

Fisher -> FILA -> LoRA -> mot tap con cac loss. KD von da tat san boi
scheme `uni_nokd`. Hai bien the:

| JOB | UU | MU | UR | MR | IHL | CE | KD |
|---|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| P3-NoKD-More (day du, run chinh) | 1/3 | 1/6 | 1/3 | 1/6 | 1 | 1 | 0 |
| `fmi4_*` — chi 4 loss Forget-MI  | 1/3 | 1/6 | 1/3 | 1/6 | **0** | **0** | 0 |
| `fmi5_*` — 4 loss + IHL          | 1/3 | 1/6 | 1/3 | 1/6 | 1 | **0** | 0 |

`fmi4` tra loi: PEFT mot minh co tai lap duoc Forget-MI khong.
`fmi5` tra loi: CE co thuc su can khong, hay IHL da du.

Moi thu khac GIU NGUYEN so voi run chinh: seed 42, 30 epoch, holdout 4 tap,
MORE (LoRA mo rong MLP + 2 khoi conv cuoi), bo chon S2. Nho vay ket qua
cam thang vao Bang 4.6 duoc.

## Da biet ve `fmi4_m3` (da chay 2026-08-05)

KHONG quen duoc: Df-AUC 0.731 = dung bang mo hinh goc, MIA_paper 0.429.
Nguyen nhan KHONG phai phan ky ma nguoc lai — qua trinh toi uu DONG BANG
tu epoch 8. `S_val` dat tot nhat o E3 roi khong cai thien, nen
ReduceLROnPlateau (factor 0.5, patience 2) chia doi lr mai cho den khi
lr ~ 4e-7. Vi khong co IHL nen G_forget cua selector gan nhu bat dong
(0.490 -> 0.468), selector khong thay tien bo -> lich lr tu tat may.

=> Voi `fmi5` thi van de nay DU KIEN khong lap lai: IHL bat nen thanh phan
   `ihl/2` trong G_forget se giam that, selector thay tien bo, lr giu duoc.
   Van nen kiem tra lai bang Cell 5.

## Sau khi chay — doc gi o Cell 5

1. `forget_ce` leo len hang chuc  -> PHAN KY vi lr. Dat LR = 1e-4 roi chay lai.
2. `d_u`, `d_m` bat dong nhieu epoch lien -> DONG BANG vi lr bi bop ve 0
   (giong `fmi4_m3`). Khong phai loi lr ban dau, ma la lich giam lr.
3. Ca hai deu binh thuong -> so dung duoc, cam vao Bang 4.6.

Doi `JOB` (va `LR` neu can) o Cell 2 roi Run All. **Moi lan chay DUNG mot job.**


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    'adv_common CHUA co hook CE-selector -> git push code MOI roi moi Save Version!'
assert os.path.exists('training/forgetmi_p3_cand.py'), 'chua push forgetmi_p3_cand.py!'
print('Code OK.')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON JOB + path discovery
import glob, os

# --- DOI 1-2 DONG NAY ---
JOB = 'fmi5_m3'   # fmi4_* = chi 4 loss Forget-MI | fmi5_* = 4 loss + IHL (bo CE)
LR  = None        # None = dung lr cua config (2e-4). Dat 1e-4 / 5e-5 neu phan ky.

SEED   = 42
EPOCHS = 30

# job -> (dataset, forget%). Tien to quyet dinh loss nao bi tat (xem EXTRA ben duoi).
JOBS = {
 'fmi4_m3' : ('mimic', 3), 'fmi5_m3' : ('mimic', 3),
 'fmi4_m6' : ('mimic', 6), 'fmi5_m6' : ('mimic', 6),
 'fmi4_m10': ('mimic',10), 'fmi5_m10': ('mimic',10),
 'fmi4_iu' : ('iu',    3), 'fmi5_iu' : ('iu',    3),
}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
DATASET, FORGET_PCT = JOBS[JOB]

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None

tag=f'{DATASET}{FORGET_PCT}per'
lrtag='' if LR is None else f'_lr{LR:g}'
RID=f'{JOB}{lrtag}_s{SEED}'
OUT=f'/kaggle/working/fmionly_{JOB}_s{SEED}'
OD=f'{OUT}/{RID}'
RESULTS=f'/kaggle/working/results_{RID}.csv'

if DATASET=='mimic':
    CONFIG = 'config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE; HAS_GOLD=bool(gh)
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG = 'config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE; HAS_GOLD=bool(reb)
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,'Khong thay iu-split / forget_set_iu'
    SPLIT=sp[0]; FORGET=fg[0]

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'results_csv_path':RESULTS,'use_noise':1}
# Cau hinh DA KHOA tu MIMIC 3%, khong tuning lai theo 6/10%/IU
MLP_TXT='attention.output.dense|intermediate.dense|output.dense'
MORE={'lora_extra_target_modules':MLP_TXT,'lora_image_last_k_blocks':2}
# *** DIEM KHAC BIET DUY NHAT giua cac bien the ***
if JOB.startswith('fmi4'):
    EXTRA={'lambda_ce':0,'lambda_ihl':0}   # chi 4 loss Forget-MI
    VARIANT='4 loss Forget-MI (tat IHL + CE)'
else:
    EXTRA={'lambda_ce':0}                  # 4 loss + IHL; IHL giu mac dinh 1.0
    VARIANT='4 loss Forget-MI + IHL (tat CE)'
if LR is not None: EXTRA['learning_rate']=LR

print('JOB',JOB,'| DATASET',DATASET,FORGET_PCT,'% | config',CONFIG,'| GOLD',HAS_GOLD)
print('BIEN THE:',VARIANT)
print('run id',RID,'| LR',('config default' if LR is None else LR))
print('EXTRA:',EXTRA)
print('BASE',BASE); print('FORGET',FORGET)


In [ ]:
# Cell 3: CHAY (30 epoch, bat CE-selector de co S2)
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

ovr=dict(COMMON); ovr.update(MORE); ovr.update(EXTRA)
ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
            'ce_selector':1,'s4_delta':0.15,
            'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
     '--scheme','uni_nokd','--ablate','none','--fresh','--override',
     ','.join(f'{k}={v}' for k,v in ovr.items())]

_exp_ihl = 0.0 if JOB.startswith('fmi4') else 1.0
print('='*72+f'\n{RID}  |  {VARIANT}\n'+'='*72)
print(f'Kiem tra dong log "Ngoai block" ngay duoi: phai la '
      f'L_CE=0.000  L_KD=0.000  L_IHL={_exp_ihl:.3f}')
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True); print(f'OK {RID}  wall {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL',RID,'rc=',e.returncode)


In [ ]:
# Cell 4: eval OG + GOLD lam moc tham chieu (bo qua neu da co tu run truoc)
import os, subprocess
RUN_REF = (FORGET_PCT!=3 or DATASET!='mimic')

def evalref(label, mpath):
    ovr=dict(COMMON); ovr['output_dir']=f'{OUT}/_ref'; ovr['results_csv_path']=RESULTS
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,
         '--method','reference','--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)

if RUN_REF:
    evalref(f'og_{tag}',BASE)
    if HAS_GOLD: evalref(f're_{tag}',GOLD)
    else: print('(khong co GOLD cho',tag,')')
else:
    print('RUN_REF=False -> OG/GOLD cua MIMIC 3% da co tu run truoc')


In [ ]:
# Cell 5: S2 + E30 + T_core + CHAN DOAN PHAN KY
import glob, json, os, pandas as pd
pd.set_option('display.width',220)

for f in sorted(glob.glob(f'{OUT}/**/selected_checkpoints.json',recursive=True)):
    d=json.load(open(f,encoding='utf-8'))['results']
    s2=d.get('S2_closest_ce',{})
    print('S2 (Closest CE) ->', os.path.basename(os.path.dirname(f)))
    if s2.get('epoch') is None:
        print('   khong chon duoc:',s2.get('note'))
    else:
        print(f"   E{s2['epoch']}  Df-AUC {s2['Df_AUC']}  Df-F1 {s2['Df_F1']}  "
              f"Dt-AUC {s2['Dt_AUC']}  Dt-F1 {s2['Dt_F1']}  MIA {s2['MIA']}  "
              f"fce {s2['forget_ce']}  nmval_ce {s2['nm_val_ce']}")

if os.path.exists(RESULTS):
    print('\n===== E30 (last) + OG/GOLD =====')
    dr=pd.read_csv(RESULTS)
    cols=[c for c in ['id','method','checkpoint_kind','checkpoint','selected_epoch',
                      'Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','forget_ce','test_ce',
                      'trainable_params','trainable_ratio'] if c in dr.columns]
    print(dr[cols].to_string(index=False))

print('\n===== T_core + GPU peak =====')
for f in sorted(glob.glob(f'{OUT}/**/timing_*.json',recursive=True)):
    d=json.load(open(f,encoding='utf-8'))
    print(f"{d.get('method'):18} T_fisher {d.get('fisher_seconds',0):7.1f}s  "
          f"T_fila {d.get('fila_seconds',0):6.1f}s  T_train {d.get('train_seconds',0):8.1f}s  "
          f"=> T_core {d.get('core_seconds',0):8.1f}s")
    print(f"{'':18} params {d.get('trainable_params',0):,} ({100*d.get('trainable_ratio',0):.2f}%)")

# --- CHAN DOAN: (a) phan ky, (b) dong bang do lich lr bop ve 0 ---
ph=f'/kaggle/working/perepoch_{RID}.csv'
if os.path.exists(ph):
    h=pd.read_csv(ph)
    print('\n===== QUY DAO =====')
    show=[c for c in ['epoch','d_u_mean','d_m_mean','ur','mr','ihl','val_ce','S_val','G_forget']
          if c in h.columns]
    print(h[show].to_string(index=False))

    # (a) phan ky
    vc=[c for c in h.columns if c in ('val_ce',)] or [c for c in h.columns if 'ce' in c.lower()]
    if vc:
        mx=h[vc[0]].max()
        print(f"\n[a] {vc[0]} lon nhat = {mx:.3f}")
        print("    *** PHAN KY -> dat LR = 1e-4 o Cell 2, chay lai. ***" if mx>10
              else "    OK, khong phai phan ky.")

    # (b) dong bang: d_u va d_m bat dong o 10 epoch cuoi
    if {'d_u_mean','d_m_mean'}.issubset(h.columns) and len(h)>=10:
        t=h.tail(10)
        rng_u=t['d_u_mean'].max()-t['d_u_mean'].min()
        rng_m=t['d_m_mean'].max()-t['d_m_mean'].min()
        print(f"[b] bien thien 10 epoch cuoi: d_u {rng_u:.4f}  d_m {rng_m:.4f}")
        if rng_u<0.05 and rng_m<0.05:
            print("    *** DONG BANG. S_val het cai thien -> ReduceLROnPlateau bop lr ve ~0.")
            print("        So do KHONG phai 'het ngan sach' ma la mot diem can bang.")
            print("        Phai ghi ro dieu nay khi bao cao.")
        else:
            print("    OK, mo hinh van dich chuyen den cuoi.")

    # (c) S_val co cai thien khong
    if 'S_val' in h.columns:
        i=int(h['S_val'].idxmin())
        print(f"[c] S_val tot nhat = {h['S_val'].min():.4f} tai epoch {int(h.loc[i,'epoch'])}"
              f" / {int(h['epoch'].max())}")
else:
    print('\n(khong thay perepoch csv)')

print('\nTAI VE: timing_*.json + selected_checkpoints.json + results_*.csv + perepoch_*.csv')
